In [ ]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.neighbors import NearestCentroid

In [ ]:
df = pd.read_csv("../wall+following+robot+navigation+data/sensor_readings_4_processed.csv")
X = df.drop(columns="Class").to_numpy()
y = df["Class"].to_numpy()
classes = np.unique(y)

SPLITS = {"20/80": 0.2, "30/70": 0.3, "50/50": 0.5, "70/30": 0.7, "80/20": 0.8}
RODADAS = 100
STATS = ["mean", "std", "max", "min", "median"]

In [ ]:
MODELOS = {"DMC robusto": NearestCentroid(metric="manhattan")}  # centroide = mediana, distância de Manhattan

In [ ]:
registros, confusoes = [], {}

for split, frac in SPLITS.items():
    for r in range(RODADAS):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, train_size=frac, stratify=y, random_state=r)
        for nome, base in MODELOS.items():
            modelo = clone(base)
            t0 = time.perf_counter()
            modelo.fit(X_tr, y_tr)
            t1 = time.perf_counter()
            pred = modelo.predict(X_te)
            t2 = time.perf_counter()

            confusoes[split, nome, r] = confusion_matrix(y_te, pred, labels=classes)
            registros.append({
                "split": split, "modelo": nome, "rodada": r,
                "acuracia": accuracy_score(y_te, pred),
                "t_treino": t1 - t0, "t_teste": t2 - t1,
                **dict(zip(classes, recall_score(y_te, pred, labels=classes, average=None))),
            })

res = pd.DataFrame(registros)

In [ ]:
grupos = res.groupby(["split", "modelo"], sort=False)
tabela = grupos["acuracia"].agg(STATS).join(grupos[["t_treino", "t_teste"]].mean())
display(tabela.round(6))

In [ ]:
plt.figure(figsize=(10, 5))
sns.pointplot(res, x="split", y="acuracia", hue="modelo", errorbar="sd", dodge=0.5 if len(MODELOS) > 1 else False)
plt.title("Acurácia (média ± desvio-padrão) por separação treino/teste");

In [ ]:
for nome in MODELOS:
    fig, axes = plt.subplots(len(SPLITS), 2, figsize=(11, 4.5 * len(SPLITS)))
    for linha, (split, sub) in zip(axes, res[res["modelo"] == nome].groupby("split", sort=False)):
        for ax, (rotulo, i) in zip(linha, [("Melhor", sub["acuracia"].idxmax()), ("Pior", sub["acuracia"].idxmin())]):
            sns.heatmap(confusoes[split, nome, sub.loc[i, "rodada"]], annot=True, fmt="d", cmap="Blues", cbar=False,
                        xticklabels=classes, yticklabels=classes, ax=ax)
            ax.set(title=f"{nome} | {split} | {rotulo} rodada (acc = {sub.loc[i, 'acuracia']:.4f})",
                   xlabel="Predito", ylabel="Real")
    plt.tight_layout()
    plt.show()

In [ ]:
por_classe = res.melt(id_vars=["split", "modelo"], value_vars=list(classes),
                      var_name="classe", value_name="acuracia_classe")

for classe, g in por_classe.groupby("classe"):
    print(classe)
    display(g.groupby(["split", "modelo"], sort=False)["acuracia_classe"].agg(STATS).round(4))

In [ ]:
sns.catplot(por_classe, x="classe", y="acuracia_classe", hue="modelo", col="split", col_wrap=2,
            kind="box", height=4, aspect=1.6);